In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import optuna
import mlflow
from sklearn.metrics import average_precision_score
import warnings
warnings.filterwarnings('ignore')

mlflow.set_tracking_uri("sqlite:///../../mlflow.db")
mlflow.set_experiment("sentinel_pickup_disruption")

df = pd.read_parquet("../../data/processed/pickup_features_v2.parquet")

MODEL_FEATURES = [
    'hour_of_day', 'courier_orders_so_far', 'day_of_week', 'accept_distance_km',
    'courier_late_rate', 'aoi_disruption_rate', 'city_hour_rate', 'courier_city_rate',
    'courier_zone_familiarity', 'courier_tenure_days', 'courier_load_3h',
    'mins_since_last_accept', 'velocity_target',
    'courier_running_rate', 'zone_running_rate', 'orders_per_courier',
    'city',
] 
CATEGORICAL = ['city', 'day_of_week', 'hour_of_day']

train = df[df['ds'] <= 950].copy()
test  = df[df['ds'] >= 951].copy()

train = train.sort_values('ds')
cutoff = train['ds'].quantile(0.85)
tr = train[train['ds'] <= cutoff]
val = train[train['ds'] > cutoff]

X_tr, y_tr = tr[MODEL_FEATURES].copy(), tr['is_disrupted']
X_val, y_val = val[MODEL_FEATURES].copy(), val['is_disrupted']
X_test, y_test = test[MODEL_FEATURES].copy(), test['is_disrupted']

for c in CATEGORICAL:
    for X in [X_tr, X_val, X_test]:
        X[c] = X[c].astype('category')

scale_pos = (y_tr == 0).sum() / (y_tr == 1).sum()
print(f"Tune-train: {X_tr.shape}, Val: {X_val.shape}, Test: {X_test.shape}")
print(f"scale_pos_weight: {scale_pos:.1f}")

/opt/anaconda3/envs/sentinel/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Tune-train: (4212112, 17), Val: (734201, 17), Test: (1118595, 17)
scale_pos_weight: 53.5


In [ ]:
def objective(trial):
    params = {
        'objective': 'binary',
        'metric': 'average_precision',
        'scale_pos_weight': scale_pos,
        'verbose': -1,
        'seed': 42,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 31, 255),
        'max_depth': trial.suggest_int('max_depth', 4, 12),
        'min_child_samples': trial.suggest_int('min_child_samples', 50, 500),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.5, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.5, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),
        'lambda_l1': trial.suggest_float('lambda_l1', 1e-8, 10.0, log=True),
        'lambda_l2': trial.suggest_float('lambda_l2', 1e-8, 10.0, log=True),
        'min_gain_to_split': trial.suggest_float('min_gain_to_split', 0.0, 1.0),
    }

    train_set = lgb.Dataset(X_tr, label=y_tr, categorical_feature=CATEGORICAL)
    val_set   = lgb.Dataset(X_val, label=y_val, categorical_feature=CATEGORICAL, reference=train_set)

    model = lgb.train(
        params, train_set,
        num_boost_round=1500,
        valid_sets=[val_set],
        callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)],
    )

    pred = model.predict(X_val, num_iteration=model.best_iteration)
    return average_precision_score(y_val, pred)

print("Objective defined. Search space covers 10 hyperparameters.")

Objective defined. Search space covers 10 hyperparameters.


In [ ]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

study = optuna.create_study(direction='maximize', study_name='lgbm_pickup')

study.optimize(objective, n_trials=30, show_progress_bar=True)

print(f"\nBest validation PR-AUC: {study.best_value:.4f}")
print(f"\nBest params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

  0%|          | 0/30 [00:00<?, ?it/s]

Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[79]	valid_0's average_precision: 0.204659


Best trial: 0. Best value: 0.204659:   3%|▎         | 1/30 [00:25<12:06, 25.05s/it]

Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[158]	valid_0's average_precision: 0.203722


Best trial: 0. Best value: 0.204659:   7%|▋         | 2/30 [00:40<09:06, 19.52s/it]

...

Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[206]	valid_0's average_precision: 0.203968


Best trial: 22. Best value: 0.206281:  97%|█████████▋| 29/30 [10:25<00:21, 22.00s/it]

Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[151]	valid_0's average_precision: 0.199851


Best trial: 22. Best value: 0.206281: 100%|██████████| 30/30 [10:37<00:00, 21.26s/it]


Best validation PR-AUC: 0.2063

Best params:
  learning_rate: 0.05510840714554918
  num_leaves: 67
  max_depth: 5
  min_child_samples: 139
  feature_fraction: 0.6466226857291427
  bagging_fraction: 0.8699915210961181
  bagging_freq: 1
  lambda_l1: 1.8730373366853883e-08
  lambda_l2: 0.08364404164743629
  min_gain_to_split: 0.010407254892571785
